In [ ]:
# ================================================================
# USER CONFIGURATION — set this path for your environment
# ================================================================
MAPS_SAVE_DIR = ''    # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# Analysis and Visualisation

This notebook produces all figures and tables for the thesis results section.
No model training occurs here — all results are loaded from CSV files
saved by the experiment notebooks.

Run order:
1. 00_setup.ipynb — download and prepare Real-IAD dataset
2. 00b_mvtec_validation.ipynb — AnomalyDINO implementation validation
3. 02_standard_protocol.ipynb — main benchmark results
4. 03_crossview_protocol.ipynb — robustness evaluation
5. 04_ablation_study.ipynb — systematic ablation investigations
6. This notebook (05_analysis.ipynb) — figures and tables

Narrative structure:
- Act 1: Standard protocol — who performs best under ideal conditions?
- Act 2: Cross-view protocol — how robust are Dinomaly and INP-Former
         to viewpoint shift? (AnomalyDINO excluded — failure mode fully
         characterised by ablation study)
- Act 3: WGA — where do models fail and is failure concentrated?
- Act 4: Ablation study — what drives observed differences?
  - Investigation 1: AnomalyDINO 2x2 factorial (multi/single class
                     x multi/single view)
  - Investigation 2: Compute equalisation between Dinomaly and INP-Former
  - Investigation 3: Cross-view volume compensation for INP-Former
- Act 5: Efficiency trade-off

In [ ]:
import os
import sys
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
results_path = f'{repo_path}/results'
figures_path = f'{repo_path}/results/figures'

os.makedirs(figures_path, exist_ok=True)

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
else:
    !git -C {repo_path} pull

sys.path.insert(0, repo_path)

!pip install anomalib==2.3.3 ADEval einops timm kornia -q

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")
wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
vis = load_module("visualisation",
                  f"{repo_path}/evaluation/visualisation.py")

compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
compute_degradation_ratio = metrics.compute_degradation_ratio

wga_by_category = wga_module.wga_by_category
wga_by_viewpoint = wga_module.wga_by_viewpoint
wga_by_defect_type = wga_module.wga_by_defect_type
print_wga_summary = wga_module.print_wga_summary
find_disagreement_groups = wga_module.find_disagreement_groups

print("All modules loaded")

In [ ]:
# Results paths — consistent across all notebooks
results_path = f'{repo_path}/results'

# Protocol-specific score paths
std_results = f'{results_path}/standard'
cv_results = f'{results_path}/crossview'
abl_results = f'{results_path}/ablation'

# Anomaly map paths
std_maps = f'{MAPS_SAVE_DIR}/standard'
cv_maps = f'{MAPS_SAVE_DIR}/crossview'
maps_abl1_mm = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_multiview'
maps_abl1_sm = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_multiview'
maps_abl1_ms = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_singleview'
maps_abl1_ss = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_singleview'
maps_abl2 = f'{MAPS_SAVE_DIR}/ablation/investigation2'
maps_abl3 = f'{MAPS_SAVE_DIR}/ablation/investigation3'

# Create all directories upfront
for path in [
    std_results,
    cv_results,
    f'{abl_results}/investigation1',
    f'{abl_results}/investigation2',
    f'{abl_results}/investigation3',
    f'{results_path}/weights',
    f'{results_path}/figures',
    std_maps,
    cv_maps,
    maps_abl1_mm,
    maps_abl1_sm,
    maps_abl1_ms,
    maps_abl1_ss,
    maps_abl2,
    maps_abl3,
]:
    os.makedirs(path, exist_ok=True)

print("All results directories ready")

In [ ]:
# Standard protocol — all three models
results_din_std = pd.read_csv(f'{std_results}/dinomaly_scores.csv')
results_dino_std = pd.read_csv(f'{std_results}/anomalydino_scores.csv')
results_inp_std = pd.read_csv(f'{std_results}/inpformer_scores.csv')

# Cross-view protocol — Dinomaly and INP-Former only
# AnomalyDINO is excluded from the cross-view protocol.
# Its failure mode (~0.50 I-AUROC on Real-IAD) is fully characterised
# by the ablation study (Investigation 1: 2x2 factorial design).
# Including it in the cross-view protocol would add no scientific value
# as its performance is already near-random on the standard protocol.
results_din_cv = pd.read_csv(f'{cv_results}/dinomaly_scores.csv')
results_inp_cv = pd.read_csv(f'{cv_results}/inpformer_scores.csv')

# Ablation study results
abl1_summary = pd.read_csv(
    f'{abl_results}/investigation1/factorial_summary.csv')
abl2_summary = pd.read_csv(
    f'{abl_results}/investigation2/compute_summary.csv')
abl3_summary = pd.read_csv(
    f'{abl_results}/investigation3/volume_summary.csv')

# Ablation investigation 1 — individual condition results
abl1_cond_a = pd.read_csv(
    f'{abl_results}/investigation1/anomalydino_multiclass_multiview_scores.csv')
abl1_cond_b = pd.read_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_multiview_scores.csv')
abl1_cond_c = pd.read_csv(
    f'{abl_results}/investigation1/anomalydino_multiclass_singleview_scores.csv')
abl1_cond_d = pd.read_csv(
    f'{abl_results}/investigation1/anomalydino_singleclass_singleview_scores.csv')

# Ablation investigation 2 and 3
abl2_inp_eq = pd.read_csv(
    f'{abl_results}/investigation2/inpformer_equalised_scores.csv')
abl3_inp_comp = pd.read_csv(
    f'{abl_results}/investigation3/inpformer_cv_compensated_scores.csv')

# Convenience dictionaries
df_dict_std = {
    'AnomalyDINO': results_dino_std,
    'Dinomaly': results_din_std,
    'INP-Former': results_inp_std,
}
df_dict_cv = {
    'Dinomaly': results_din_cv,
    'INP-Former': results_inp_cv,
}

print("All results loaded")
print(f"Standard protocol models: {list(df_dict_std.keys())}")
print(f"Cross-view protocol models: {list(df_dict_cv.keys())}")
print(f"Standard test images: {len(results_din_std)}")
print(f"Cross-view test images: {len(results_din_cv)}")

# ── Map loading helpers ───────────────────────────────────────

def load_anomaly_map(image_path, model_name, protocol='standard'):
    """
    Load anomaly map and score for a given image and model.

    Args:
        image_path:  full path to the original image
        model_name:  'Dinomaly', 'AnomalyDINO', or 'INP-Former'
        protocol:    'standard', 'crossview', or ablation sub-path

    Returns:
        anomaly_map: numpy array (H, W) or None if not found
        score:       float or None if not found
    """
    if not MAPS_SAVE_DIR:
        print("MAPS_SAVE_DIR not set — cannot load anomaly maps")
        return None, None

    stem = os.path.splitext(os.path.basename(image_path))[0]
    parts = image_path.replace('\\', '/').split('/')
    try:
        category = parts[-3]
    except IndexError:
        category = 'unknown'

    npz_path = os.path.join(
        MAPS_SAVE_DIR, protocol, model_name, category, f"{stem}.npz")

    if not os.path.exists(npz_path):
        return None, None

    data = np.load(npz_path)
    return data['anomaly_map'], float(data['anomaly_score'])


def load_maps_for_image(image_path, models, protocol='standard'):
    """
    Load anomaly maps for all models for a given image.
    Returns dict of model_name -> (anomaly_map, score).
    Only includes models where the map file exists.
    """
    result = {}
    for model_name in models:
        amap, score = load_anomaly_map(image_path, model_name, protocol)
        if amap is not None:
            result[model_name] = (amap, score)
    return result


print("\nMap loading helpers ready")
print(f"Maps directory: {MAPS_SAVE_DIR if MAPS_SAVE_DIR else 'NOT SET'}")

## Act 1: Standard Protocol

Research Question 1: Which DINOv2-based detection paradigm achieves the
highest anomaly detection performance under standard multi-view evaluation
conditions on Real-IAD?

All three models are evaluated: AnomalyDINO (memory-based),
Dinomaly (reconstruction-based), and INP-Former (prototype-based).
This establishes the baseline performance ranking and reveals
AnomalyDINO's fundamental limitation on multi-view data.

In [ ]:
print("Computing standard protocol metrics...")

std_metrics = {}
for model_name, df in df_dict_std.items():
    m = compute_all_metrics(df)
    std_metrics[model_name] = m

std_summary = pd.DataFrame(std_metrics).T.reset_index()
std_summary.columns = ['Model'] + list(std_summary.columns[1:])

print("\n" + "="*65)
print("TABLE 1: Standard Protocol Results — Real-IAD (All 30 Categories)")
print("="*65)
print(std_summary.round(4).to_string(index=False))

std_summary.to_csv(
    f'{results_path}/table1_standard_summary.csv', index=False)
print(f"\nSaved to results/table1_standard_summary.csv")

In [ ]:
vis.plot_score_distributions(
    df_dict_std,
    output_path=f'{figures_path}/fig1_score_distributions_standard.png'
)
print("Figure 1 saved: score distributions (standard protocol, all models)")

In [ ]:
vis.plot_per_category_comparison(
    df_dict_std,
    metric_fn=compute_i_auroc,
    metric_name='I-AUROC',
    output_path=f'{figures_path}/fig2_per_category_standard.png'
)
print("Figure 2 saved: per-category I-AUROC (standard protocol)")

In [ ]:
vis.plot_wga_category_table(
    df_dict_std,
    output_path=f'{figures_path}/fig3_wga_category_table_standard.png'
)
print("Figure 3 saved: WGA category table (standard protocol)")

In [ ]:
vis.plot_wga_heatmap_viewpoint(
    df_dict_std,
    output_path=f'{figures_path}/fig4_wga_heatmap_viewpoint_standard.png'
)
print("Figure 4 saved: WGA heatmap category x viewpoint (standard protocol)")

In [ ]:
# Top 5 hardest defect types per model on standard protocol
print("="*60)
print("HARDEST DEFECT TYPES — Standard Protocol")
print("="*60)

all_defect_wga = []
for model_name, df in df_dict_std.items():
    wga_df = wga_by_defect_type({model_name: df})
    if not wga_df.empty:
        worst = wga_df.nsmallest(5, 'auroc')[
            ['defect_type', 'n_anomalous', 'auroc']]
        print(f"\n{model_name} — 5 hardest defect types:")
        print(worst.round(4).to_string(index=False))
        wga_df['model'] = model_name
        all_defect_wga.append(wga_df)

if all_defect_wga:
    defect_table = pd.concat(all_defect_wga)
    defect_table.to_csv(
        f'{results_path}/table_defect_type_wga_standard.csv',
        index=False)
    print("\nSaved defect type WGA table")

In [ ]:
# Generate disagreement figures for standard protocol
from evaluation.wga import find_disagreement_groups

disagreement_df = find_disagreement_groups(df_dict_std, top_n=10)

figures_generated = []
for idx, row in disagreement_df.reset_index().iterrows():
    category = row.get('category', '')
    viewpoint = row.get('viewpoint', '')
    defect_type = row.get('defect_type', '')

    # Find representative image — highest scoring anomalous image
    # from the best performing model for this group
    best_model = max(
        df_dict_std.keys(),
        key=lambda m: row.get(m, 0) if not pd.isna(row.get(m, 0)) else 0
    )
    best_df = df_dict_std[best_model]
    group_mask = (
        (best_df['category'] == category) &
        (best_df['viewpoint'] == viewpoint) &
        (best_df['defect_type'] == defect_type) &
        (best_df['label'] == 1)
    )
    group_images = best_df[group_mask].sort_values(
        'image_score', ascending=False)

    if group_images.empty:
        continue

    image_path = group_images.iloc[0]['image_path']

    # Load maps for all models
    maps_data = load_maps_for_image(
        image_path,
        models=list(df_dict_std.keys()),
        protocol='standard'
    )

    if not maps_data:
        print(f"No maps found for group {idx+1} — skipping")
        continue

    amaps = {m: v[0] for m, v in maps_data.items()}
    scores_per_model = {m: v[1] for m, v in maps_data.items()}

    # Load GT mask
    first_df = list(df_dict_std.values())[0]
    img_row = first_df[first_df['image_path'] == image_path]
    gt_mask = None
    if not img_row.empty and img_row.iloc[0].get('has_mask'):
        mask_path = img_row.iloc[0].get('mask_path', '')
        if mask_path and Path(mask_path).exists():
            from PIL import Image as PILImage
            gt_mask = np.array(
                PILImage.open(mask_path).convert('L')) / 255.0

    output_path = (
        f"{figures_path}/disagreement_{idx+1}_"
        f"{category}_{viewpoint}_{defect_type}_"
        f"{Path(image_path).stem}.png"
    )

    vis.plot_anomaly_map_comparison(
        image_path=image_path,
        anomaly_maps=amaps,
        gt_mask=gt_mask,
        title=(f"Disagreement Group {idx+1}: "
               f"{category} | {viewpoint} | {defect_type}"),
        output_path=output_path
    )
    figures_generated.append(output_path)

print(f"\nGenerated {len(figures_generated)} disagreement figures")
print("Saved to results/figures/")

## Act 1 Ablations: Standard Protocol Investigations

Two ablation investigations address questions raised by the standard
protocol results:

Investigation 1 (AnomalyDINO 2x2 Factorial):
AnomalyDINO achieves near-random I-AUROC (~0.50) on Real-IAD.
A 2x2 factorial design isolates whether this is due to intra-class
viewpoint variation (H1) or cross-category feature contamination (H2),
or both. Evaluated on five representative categories.

Investigation 2 (Compute Equalisation):
Dinomaly and INP-Former use different training conventions resulting
in a ~9x disparity in total image passes. This investigation equalises
compute budgets to assess whether performance differences reflect
architecture or training volume.

In [ ]:
print("="*60)
print("INVESTIGATION 1: AnomalyDINO 2x2 FACTORIAL")
print("="*60)
print(abl1_summary.round(4).to_string(index=False))

print("\nInterpretation guide:")
print("  Viewpoint Effect = Cond A minus Cond C")
print("  (how much multi-view hurts vs single-view)")
print("  Contamination Effect = Cond A minus Cond B")
print("  (how much multi-class hurts vs single-class)")
print("  If viewpoint effect > contamination effect: H1 confirmed dominant")

abl1_summary.to_csv(
    f'{results_path}/ablation1_factorial_summary.csv', index=False)

In [ ]:
print("="*60)
print("INVESTIGATION 2: COMPUTE EQUALISATION")
print("Dinomaly vs INP-Former at matched ~3M image passes")
print("="*60)
print(abl2_summary.round(4).to_string(index=False))

vis.plot_ablation_compute(
    abl2_summary,
    output_path=f'{figures_path}/fig5_ablation_compute_equalisation.png'
)
print("\nFigure 5 saved: compute equalisation ablation")

## Act 2: Cross-View Protocol

Research Question 2: How robust are reconstruction-based and prototype-based
DINOv2 models to viewpoint shift when deployed on camera angles not seen
during training?

AnomalyDINO is excluded from this protocol. Its performance on the
standard protocol is already near-random and its failure mode is fully
characterised by Investigation 1. Including it here would not add
scientific value.

Dinomaly and INP-Former are trained on C1 and C2 viewpoints only and
evaluated on the unseen C3, C4, and C5 viewpoints. The degradation
ratio quantifies the cost of viewpoint shift for each model.

In [ ]:
print("Computing cross-view protocol metrics...")

cv_metrics = {}
for model_name, df in df_dict_cv.items():
    m = compute_all_metrics(df)
    cv_metrics[model_name] = m

cv_summary = pd.DataFrame(cv_metrics).T.reset_index()
cv_summary.columns = ['Model'] + list(cv_summary.columns[1:])

print("\n" + "="*65)
print("TABLE 2: Cross-View Protocol Results — Real-IAD")
print("="*65)
print(cv_summary.round(4).to_string(index=False))

cv_summary.to_csv(
    f'{results_path}/table2_crossview_summary.csv', index=False)
print(f"\nSaved to results/table2_crossview_summary.csv")

In [ ]:
std_auroc = {m: compute_i_auroc(df) for m, df in df_dict_std.items()}
cv_auroc = {m: compute_i_auroc(df) for m, df in df_dict_cv.items()}

# Compare only models present in both protocols
vis.plot_performance_comparison(
    results_std={m: std_auroc[m] for m in df_dict_cv.keys()},
    results_cv=cv_auroc,
    metric='I-AUROC',
    output_path=f'{figures_path}/fig6_performance_comparison_cv.png'
)
print("Figure 6 saved: standard vs cross-view I-AUROC comparison")

In [ ]:
degradation = {
    m: compute_degradation_ratio(std_auroc[m], cv_auroc[m])
    for m in df_dict_cv.keys()
}

print("\nPerformance Degradation Ratios (Dinomaly and INP-Former):")
for model, deg in degradation.items():
    print(f"  {model}: {deg:.2f}%")

vis.plot_degradation_ratios(
    degradation,
    output_path=f'{figures_path}/fig7_degradation_ratios.png'
)
print("\nFigure 7 saved: degradation ratios")

In [ ]:
vis.plot_wga_heatmap_viewpoint(
    df_dict_cv,
    output_path=f'{figures_path}/fig8_wga_heatmap_viewpoint_crossview.png'
)
print("Figure 8 saved: WGA category x viewpoint (cross-view protocol)")
print("Dinomaly and INP-Former only")

In [ ]:
from evaluation.wga import compute_wga

delta_data = {}
for model_name in df_dict_cv.keys():
    std_wga = compute_wga(df_dict_std[model_name], ['category'])
    cv_wga = compute_wga(df_dict_cv[model_name], ['category'])
    std_cat = std_wga.set_index('category')['auroc']
    cv_cat = cv_wga.set_index('category')['auroc']
    delta = (cv_cat - std_cat).dropna()
    delta_data[model_name] = delta

delta_df = pd.DataFrame(delta_data)
delta_df = delta_df.sort_values(delta_df.columns[0], ascending=True)

print("\nPer-category AUROC delta (cross-view minus standard):")
print("Negative values = performance drops under viewpoint shift")
print(delta_df.round(4).to_string())

delta_df.to_csv(
    f'{results_path}/table_delta_wga_crossview_vs_standard.csv')
print("\nSaved delta WGA table")

## Act 2 Ablation: Cross-View Volume Compensation

Investigation 3 (INP-Former only):
The cross-view protocol reduces INP-Former's total image passes by ~2.5x
due to its epoch-based training schedule. Dinomaly is unaffected — its
iteration-based schedule processes fixed total passes regardless of
dataset size. This investigation compensates INP-Former's epochs
proportionally to match standard protocol total image passes,
testing whether the cross-view degradation is attributable to
reduced training volume or to viewpoint coverage itself.

In [ ]:
print("="*60)
print("INVESTIGATION 3: CROSS-VIEW VOLUME COMPENSATION (INP-Former)")
print("="*60)
print(abl3_summary.to_string(index=False))

# Extract INP-Former row for plot
inp_abl3 = abl3_summary[
    abl3_summary['Model'] == 'INP-Former'].reset_index(drop=True)

if not inp_abl3.empty:
    vis.plot_ablation_compute(
        inp_abl3,
        output_path=f'{figures_path}/fig9_ablation_volume_inpformer.png'
    )
    print("\nFigure 9 saved: INP-Former cross-view volume compensation")

print("\nNote on Dinomaly:")
din_row = abl3_summary[abl3_summary['Model'] == 'Dinomaly']
if not din_row.empty:
    print(din_row.to_string(index=False))
    print("Dinomaly degradation is purely due to reduced viewpoint")
    print("diversity — its total image passes are unchanged.")

## Act 3: Efficiency Analysis

Research Question 3: What are the computational trade-offs between the
three detection paradigms in terms of inference latency and memory footprint?

AnomalyDINO is included here despite its poor detection performance,
as its efficiency profile is still relevant for deployment decisions.
It represents the training-free lower bound of computational cost.

In [ ]:
timing_file = f'{results_path}/inference_timing.csv'

if Path(timing_file).exists():
    timing_df = pd.read_csv(timing_file)
    timing_dict = dict(zip(timing_df['model'], timing_df['mean_ms']))

    auroc_dict = {m: compute_i_auroc(df)
                  for m, df in df_dict_std.items()}

    vis.plot_efficiency_tradeoff(
        timing_dict=timing_dict,
        auroc_dict=auroc_dict,
        output_path=f'{figures_path}/fig10_efficiency_tradeoff.png'
    )
    print("Figure 10 saved: accuracy vs inference time trade-off")

    print("\nAccuracy-to-Latency Ratio (I-AUROC / ms):")
    for model in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
        if model in timing_dict and model in auroc_dict:
            ratio = auroc_dict[model] / timing_dict[model]
            print(f"  {model}: {ratio:.4f}")
else:
    print("Timing results not yet available.")
    print("Run measure_inference_time() for each model in")
    print("02_standard_protocol.ipynb and save to results/inference_timing.csv")

## Act 4: Synthesis

Bringing together all findings to answer the three research questions
and make a conclusive statement about which detection paradigm is most
suitable for multi-view industrial anomaly detection on Real-IAD.

In [ ]:
print("="*70)
print("COMBINED RESULTS SUMMARY")
print("="*70)

combined_rows = []
for model_name in ['AnomalyDINO', 'Dinomaly', 'INP-Former']:
    std_auroc_val = compute_i_auroc(df_dict_std[model_name])
    s_auroc_val = compute_s_auroc(df_dict_std[model_name])

    # Cross-view only available for Dinomaly and INP-Former
    if model_name in df_dict_cv:
        cv_auroc_val = compute_i_auroc(df_dict_cv[model_name])
        deg = compute_degradation_ratio(std_auroc_val, cv_auroc_val)
    else:
        cv_auroc_val = 'N/A'
        deg = 'N/A'

    combined_rows.append({
        'Model': model_name,
        'Paradigm': {
            'AnomalyDINO': 'Memory-Based',
            'Dinomaly': 'Reconstruction-Based',
            'INP-Former': 'Prototype-Based'
        }[model_name],
        'I-AUROC (Std)': round(std_auroc_val, 4),
        'S-AUROC (Std)': round(s_auroc_val, 4),
        'I-AUROC (CV)': round(cv_auroc_val, 4)
            if isinstance(cv_auroc_val, float) else cv_auroc_val,
        'Degradation (%)': round(deg, 2)
            if isinstance(deg, float) else deg,
    })

combined_df = pd.DataFrame(combined_rows)
print(combined_df.to_string(index=False))

combined_df.to_csv(
    f'{results_path}/table_combined_summary.csv', index=False)
print(f"\nSaved to results/table_combined_summary.csv")

In [ ]:
print("\n" + "="*60)
print("FIGURE INDEX — All thesis figures")
print("="*60)

figures = [
    ("fig1_score_distributions_standard.png",
     "Score distributions: normal vs anomalous, all three models, "
     "standard protocol"),
    ("fig2_per_category_standard.png",
     "Per-category I-AUROC comparison, standard protocol"),
    ("fig3_wga_category_table_standard.png",
     "WGA category table with colour scale, standard protocol"),
    ("fig4_wga_heatmap_viewpoint_standard.png",
     "WGA heatmap: category x viewpoint, standard protocol"),
    ("fig5_ablation_compute_equalisation.png",
     "Investigation 2: Dinomaly vs INP-Former compute equalisation"),
    ("fig6_performance_comparison_cv.png",
     "I-AUROC: standard vs cross-view, Dinomaly and INP-Former"),
    ("fig7_degradation_ratios.png",
     "Performance degradation ratios under viewpoint shift"),
    ("fig8_wga_heatmap_viewpoint_crossview.png",
     "WGA heatmap: category x viewpoint, cross-view protocol"),
    ("fig9_ablation_volume_inpformer.png",
     "Investigation 3: INP-Former cross-view volume compensation"),
    ("fig10_efficiency_tradeoff.png",
     "Accuracy vs inference time trade-off, all three models"),
]

for fname, description in figures:
    full_path = f'{figures_path}/{fname}'
    exists = Path(full_path).exists()
    status = "READY" if exists else "PENDING"
    print(f"  [{status}] {fname}")
    print(f"         {description}")

print(f"\nAll figures saved to: {figures_path}")
print("Note: Investigation 1 (AnomalyDINO 2x2 factorial) is presented")
print("as a table in the thesis — see results/ablation1_factorial_summary.csv")